# 🔥 TCN Training - Kaggle

Notebook per addestrare la TCN con ottimizzazione Optuna.

## 1. 📦 Clone Repository

In [ ]:
# Clone del repository dal branch training_branch
!git clone --branch training_branch --single-branch https://github.com/scorzaluca/Progetto_deep_learning.git
!ls Progetto_deep_learning/

In [ ]:
%pip install optuna -q

import sys
sys.path.insert(0, '/kaggle/working/Progetto_deep_learning')
print("Repository pronto!")

## 2. ⚙️ Configurazione

In [ ]:
import torch

# ===== CONFIGURAZIONE =====
DATA_PATH = "/kaggle/input/your-dataset/preprocessed_ds.csv"  # <-- MODIFICA QUESTO

MODEL_NAME = "tcn"
N_TRIALS = 50
N_FOLDS = 1        # 1=screening, 3=intensive
TUNING_EPOCHS = 50
PATIENCE = 10

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch: {torch.__version__}")
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 3. 📊 Carica Dati

In [ ]:
from src.config import SEED
from src.Utils import set_seed, load_data_and_folds

set_seed(SEED)

# Carica dataset e crea folds (passa il path Kaggle!)
df, folds = load_data_and_folds(data_path=DATA_PATH)
print(f"\nDataset: {df.shape}")
print(f"Folds: {len(folds)}")

## 4. 🔍 Ottimizzazione Optuna

In [ ]:
from src.Tuning import OptunaOptimizer

config = {
    "n_trials": N_TRIALS,
    "n_folds": N_FOLDS,
    "epochs": TUNING_EPOCHS,
    "patience": PATIENCE,
}

optimizer = OptunaOptimizer(
    model_name=MODEL_NAME,
    folds=folds,
    device=DEVICE,
    config=config,
    verbose=False,
)

result = optimizer.optimize(
    study_name=f"{MODEL_NAME}_kaggle",
    n_trials=N_TRIALS,
    new_study=True,
)

print(f"\nBest MASE: {result['best_mase']:.4f}")
print(f"Best params: {result['best_params']}")

## 5. 🏋️ Final Training

In [ ]:
from src.Training.engine import create_model, fit_model
from src.DataLoading import create_final_train_val_loaders
from src.config import TARGET_COL, NAIVE_MAE_FINAL_FOLD

best_params = result["best_params"]

# Final loaders
train_loader, val_loader, scaler = create_final_train_val_loaders(df, TARGET_COL)

# Create model
model = create_model(MODEL_NAME, best_params)
model.to(DEVICE)

# Train
model, history, best_epoch = fit_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=TUNING_EPOCHS,
    lr=best_params.get("lr", 0.001),
    device=DEVICE,
    patience=PATIENCE,
    grad_clip_norm=best_params.get("grad_clip_norm", 1.0),
    verbose=True,
    baseline_mae=NAIVE_MAE_FINAL_FOLD,
)

print(f"\nBest MASE finale: {min(history['val_mase']):.4f}")

## 6. 💾 Salva Risultati

In [ ]:
import json

# Checkpoint
torch.save(model.state_dict(), f"{MODEL_NAME}_best_model.pth")

# Params
with open(f"{MODEL_NAME}_best_params.json", "w") as f:
    json.dump(best_params, f, indent=2)

# History
with open(f"{MODEL_NAME}_history.json", "w") as f:
    json.dump(history, f, indent=2)

print("Salvati: model, params, history")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history["train_loss"], label="Train")
axes[0].plot(history["val_loss"], label="Val")
axes[0].set_title("MSE Loss")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history["val_mase"], color="green")
axes[1].axhline(y=1.0, color="red", linestyle="--")
axes[1].set_title("MASE")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f"{MODEL_NAME}_training.png", dpi=150)
plt.show()